# Revised Ground Truth 
- I have also decided a more principled way of doing the model comparison would be to fit Fourier modes to the data and use an MCMC to extract the uncertainties.
---
- Use the found priors as initial conditions.
- Randomise this.
- Opt with Fourier mode of 3 with MCMC.
- Define the samples with a weighted likelihood.
- Predict and extract posterios samples.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import emcee

from scipy.optimize import minimize
from scipy.special import softmax

import celerite2
from celerite2 import terms

from prettytable import PrettyTable

from scipy.stats import norm

import sys
from pathlib import Path
from IPython.display import display as ipy_display

sys.path.append(str(Path('../..').resolve()))

from helpers.df_ops import prepare_df, split_df, clean_df
from helpers.priors import find_classify_signals, get_priors

def downsample_min_gap(df, minimum_gap):
    days = df["day"].values
    keep = np.zeros(len(days), dtype=bool)
    keep[0] = True
    last_kept = days[0]
    for i in range(1, len(days)):
        if days[i] - last_kept >= minimum_gap:
            keep[i] = True
            last_kept = days[i]
    return df[keep].reset_index(drop=True)



In [2]:
# ---- PARAMS ----
datapath            = r"../../Data/benchmark/HD81809_Mt_wilson_data.txt"
direct_bound_tol    = 0.1
sm2016_bound_tol    = 0.2
mean_bound_tol      = 1
add_prefix          = False
relative            = True
train_split         = 0.6
valid_split         = 0.15 
star_type           = "G"
star_name           = "HD201091"
error_percent       = 2.5
sigma_upper_mult    = 5.0
q_bounds_in         = [1, 5]  # THIS HAS BEEN CHANGED
verbose             = True
plot                = True
loop_verbose        = True
loop_plot           = True
loop_savefigs       = False
sampler_plot        = True
result_plot_cadence = 50
result_plot_extra   = 1000
require_mid         = True
total_sample_count  = 2500
plot_every          = 200
subsample           = 500
n_walkers           = 32
valid_metric        = "CRPS"
SEED                = 1701


verbose_clean = False
plot_clean = False
plot_fitpeaks = False
verbose_fitpeaks = False
plot_genpriors = False
verbose_genpriors = False
plot_comparison = False
verbose_comparison = False
verbose_selected = True
plot_sampler = False


pred_forward_years = 2

low_cad = 30
high_cad = 10

min_gap = 3

In [ ]:
raw_df  = pd.read_csv(datapath, sep=r'\s+', skip_blank_lines=True)
raw_data_df  = prepare_df(raw_df, add_prefix=True, relative=True)

# Downsample
data_df = downsample_min_gap(raw_data_df, min_gap)

#Split
dirty_train_df, dirty_valid_df, test_df = split_df(data_df, train_split=train_split, valid_split=valid_split)

# Clean
train_df, valid_df, MAD = clean_df(dirty_train_df, dirty_valid_df, tol=4, verbose=verbose_clean, plot=plot_clean)

In [10]:
raw_std = np.std(raw_data_df['sind'])

log_AB_bounds = np.log([[1e-4, 5*raw_std] for _ in range(6)])  # (6,2) 
log_T_bounds  = np.log([[1e-4, raw_data_df['day'].iloc[-1] - raw_data_df['day'].iloc[0]]])  # (1,2)
c_bounds = [[raw_data_df['sind'].min(), raw_data_df['sind'].max()]]  # (1,2)

bounds = np.concatenate([log_AB_bounds, log_T_bounds, c_bounds])


def lnPost_Fourier(params, raw_data_df, t0_day, error_percent, tau, bounds):
    '''
    Takes in the retraining and test sets.
    Return the likelihood of some set of parameters.
    Based on a 3 mode fitting with A1, A2, A3, B1, B2, B3, T, and c
    The df here should not be downsampled

    Params[-1] is linear and the rest is log

    Tau should be rho_priors['mid']
    '''
    A1, A2, A3 = np.exp(params[0:3])
    B1, B2, B3 = np.exp(params[3:6])
    T = np.exp(params[6])
    c = params[7]

    # If outside bounds then must return -np.inf
    if np.any(params < bounds[:, 0]) or np.any(params > bounds[:, 1]):
        return -np.inf
    
    # No need for a correction term because loguniform priors are sampled in logspace and uniform priors are sampled in uniform space
    
    # prior_correction = 0 
    # Loguniform correction term
    # for param, bound in zip(params[0:6], bounds[0:6]):
    #     upper = bound[1]
    #     lower = bound[0]
    #     prior_correction += -np.log(param) - np.log(np.log(upper / lower))

    # Now define the lnL
    t_days = raw_data_df['day'].to_numpy()
    weights = np.where(t_days < t0_day, np.exp(-(t0_day - t_days) / tau), 1.0)
    # weights = [np.exp(-(t0_day - t_day)/tau) if t_day < t0_day else 1 for t_day in t_days]

    # weights_test = [1] * len(test_df) # constant weights


    # t0_day = test_df['day'].iloc[0] # start of the predictions
    # t_retraining = retraining_df['day']
    # t_delta = t0_day - t_retraining
    # weights_retraining = np.exp(-t_delta/tau) # downweight earlier samples

    # weights = np.concatenate([weights_retraining, weights_test])

    # Gaussian likelihood func
    # full_df = retraining_df.join(test_df)
    full_df = raw_data_df
    N = len(full_df)
    t = full_df['day']
    sigma = np.std(full_df['sind']) * error_percent / 100     # Take sigma to be func of error_percent

    y = full_df['sind']
    model = c + A1*np.sin(2*np.pi*t/T) + B1*np.cos(2*np.pi*t/T)
    model += A2*np.sin(4*np.pi*t/T) + B2*np.cos(4*np.pi*t/T)
    model += A3*np.sin(6*np.pi*t/T) + B3*np.cos(6*np.pi*t/T)
    lnL = -N * np.log(sigma) - N/2 * np.log(2*np.pi) - np.sum(((1/(2 * sigma **2)) * (y-model)**2) * weights)

    return lnL 


In [7]:
#---LSP--- the mid prior is used as tau and for the initial guess of T
classified_signal_data = find_classify_signals(raw_data_df,
                                            plot_fitpeaks=plot_fitpeaks, verbose_fitpeaks=verbose_fitpeaks,
                                            plot_genpriors=plot_genpriors, verbose_genpriors=verbose_genpriors)

rho_priors, rho_prior_bounds, rho_sources = get_priors(classified_signal_data,
                                                    star_type=star_type,
                                                    direct_bound_tol=direct_bound_tol,
                                                    sm2016_bound_tol=sm2016_bound_tol,
                                                    mean_bound_tol=mean_bound_tol,
                                                    verbose=verbose)

+------------+------------+-------------+----------------------+-----------------+--------+
| Cycle Type | Prior Days | Prior Years |     Bounds Days      |   Bounds Years  | Source |
+------------+------------+-------------+----------------------+-----------------+--------+
|   Short    |   37.33    |     0.10    |    (33.60, 41.06)    |   (0.09, 0.11)  | found  |
|    Mid     |  2923.04   |     8.01    |  (2630.74, 3215.35)  |   (7.21, 8.81)  | found  |
|    Long    |  36500.00  |    100.00   | (18250.00, 54750.00) | (50.00, 150.00) |  mean  |
+------------+------------+-------------+----------------------+-----------------+--------+


In [ ]:
raw_initial_guess_AB = [(max(raw_data_df['sind']) - min(raw_data_df['sind'])) / 6 for _ in range(6)] #these are to be sampled i nlog space
raw_initial_guess_T = [rho_priors['mid']]
initial_guess_c = [np.median(raw_data_df['sind'])]

initial_guesses = np.concatenate([np.log(raw_initial_guess_AB), np.log(raw_initial_guess_T), initial_guess_c])

walker_start_coords = initial_guesses + 1e-4 * np.random.randn(n_walkers, len(initial_guesses))

tau = rho_priors['mid']
t0_day = test_df['day'].iloc[0]

sampler = emcee.EnsembleSampler(n_walkers, len(initial_guesses), lnPost_Fourier,
                args=(raw_data_df, t0_day, error_percent, tau, bounds))
sampler.run_mcmc(walker_start_coords, nsteps=total_sample_count, progress=True)

 31%|███       | 777/2500 [01:35<03:31,  8.16it/s]